# P043: 顧客マスタの品質上の問題を探す

Level 3 ・ 探索(EDA) ・ 目安 10 分

**解答と解説**

In [ ]:
import pandas as pd

df = pd.read_csv("data/members_raw.csv")

## 1. 推奨確認項目

1. 全体: `shape`, `head()`, `dtypes`
2. 欠損: `isna().sum()`
3. 重複: `duplicated().sum()` と `["member_id"].duplicated().sum()`。重複している ID の行を見る
4. 数値の範囲: `describe()`、怪しい値の行を条件で抽出
5. カテゴリ: 各列の `value_counts()`
6. 日付: `min()` / `max()`

## 2. 模範的なコード

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum(), df["member_id"].duplicated().sum()

In [ ]:
# 完全重複ではないのに ID が重複している行
dup_ids = df[df["member_id"].duplicated()]["member_id"]
df[df["member_id"].isin(dup_ids)].sort_values("member_id")

In [ ]:
df.describe()

In [ ]:
df[(df["age"] < 0) | (df["age"] > 120)]

In [ ]:
df["gender"].value_counts()

In [ ]:
df["prefecture"].value_counts()

In [ ]:
df["plan"].value_counts()

In [ ]:
df["signup_date"].min(), df["signup_date"].max()

## 3. このデータで観察すべきポイント

| 問題 | 内容 |
|---|---|
| 欠損 | `age` 19件、`prefecture` 9件 |
| 完全重複行 | 3行(M0101〜M0103) |
| **ID だけ重複** | M0021, M0022 が2行ずつあり、`plan` が free と premium で食い違っている。どちらが正しいか確認が必要(完全重複ではないので `drop_duplicates()` では消えない) |
| 異常値 | `age` に **999 が2件、-1 が1件**。平均 54.7 に対して中央値 48、std 100 と describe からも分かる |
| 表記ゆれ | `gender`: M / F に加えて male, female, 女性。`prefecture`: tokyo, TOKYO, `"Tokyo "`(末尾に空白) |
| 型 | `monthly_spend` が `str`。`"7,620"` のようにカンマ付きの文字列なので、describe に出てこない |
| 日付 | `signup_date` の最大が **2026-12-01**。他は 2025-03 までなので、入力ミスの可能性 |

表記ゆれの統一やカンマ付き文字列の数値化は、ここで見つけられれば十分です。変換処理は AI に書かせて構いません。

## 4. 今回覚えるポイント

- 品質チェックの型: **欠損 → 重複(行とID) → 数値の範囲 → カテゴリの値 → 型 → 日付の範囲**
- 「ID だけ重複」は完全重複より厄介。`df[df["id"].isin(重複ID)]` で中身を見比べる
- カテゴリ列は `value_counts()` を1列ずつ見るのが一番早い
- 「数値のはずの列が describe に出てこない」= 型がおかしいサイン